# Основная модель

Откройте этот ноутбук в Jupyter и запустите все ячейки через Run All. Здесь проверяем качество на `verified.csv`, обучаем обе модели на доступной разметке и сохраняем прогнозы в `models/submission.csv`.

Вероятность сделки — 75% прогноза L2 с дополнительными метками из переписки и 25% прогноза L2 на проверенных ответах. Порог выбираем по сумме баллов ТЗ за F1 и ошибку долей по категориям. Заблокированным объявлениям присваиваем `is_deal = 0`.

## 1. Загрузить данные

Используем `features.csv` из `feature/feature_engineering.ipynb`. Метки и признаки сопоставляем по `listing_id`.

In [ ]:
import os
from pathlib import Path

repo_root = Path.cwd()
if repo_root.name in {"analysis", "feature", "models"}:
    repo_root = repo_root.parent

assert (repo_root / "requirements.txt").is_file() and (repo_root / "data/listings.csv").is_file(), (
    "Запустите ноутбук из корня репозитория или из папки, в которой он лежит"
)
os.chdir(repo_root)

In [13]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

In [14]:
FEATURES_PATH = Path("data_by_dewallen/features.csv")
VERIFIED_PATH = Path("data/verified.csv")
EXTENDED_VERIFIED_PATH = Path("analysis/extend_verified.csv")
LISTINGS_PATH = Path("data/listings.csv")
OUTPUT_PATH = Path("models/submission.csv")

CHAT_MODEL_SHARE = 0.75
CHAT_LABEL_WEIGHT = 0.10
CHAT_C = 0.3
VERIFIED_C = 1.0

CV_SEEDS = [11, 23, 42, 71, 101, 13, 29, 59, 83, 107]
THRESHOLDS = np.round(np.arange(0.20, 0.71, 0.01), 2)
INNER_SEED = 42

In [15]:
features = pd.read_csv(FEATURES_PATH, index_col="listing_id")
verified = pd.read_csv(VERIFIED_PATH, usecols=["listing_id", "is_deal"], index_col="listing_id")
listings = pd.read_csv(LISTINGS_PATH, usecols=["listing_id", "close_type"]).drop_duplicates()
listings = listings.set_index("listing_id")

assert listings.index.is_unique and features.index.is_unique
assert set(features.index) == set(listings.index), "Пересчитайте признаки для всех объявлений"
assert verified.index.is_unique, "Повторяются ID в проверенной разметке"
assert verified.index.isin(features.index).all()
assert verified["is_deal"].isin([0, 1]).all(), "Некорректные метки в verified.csv"
assert features.columns.is_unique

feature_columns = features.columns.difference(["seller_id", "is_deal", "weight"], sort=False)
X = features[feature_columns]
y = verified["is_deal"]
categories = features.loc[y.index, "category"]
assert categories.notna().all(), "Для CatShareMAE нужна категория каждого проверенного объявления"

print(f"Объявлений в матрице: {len(X)}")
print(f"Признаков: {X.shape[1]}")
print(f"Объявлений с проверенными метками: {len(y)}")

Объявлений в матрице: 15274
Признаков: 132
Объявлений с проверенными метками: 300


### Дополнительные метки из переписки

Берём строки с исходным весом 1.0 из `analysis/extend_verified.csv`, которые отсутствуют в `verified.csv`. В этой версии разметки это 614 меток из переписки. В обучении их вес равен 0.10. Проверенные ответы обе модели получают с весом 1.0.

In [16]:
extended = pd.read_csv(
    EXTENDED_VERIFIED_PATH,
    usecols=["listing_id", "is_deal", "weight"],
    index_col="listing_id",
)
assert extended.index.is_unique, "Повторяются ID в расширенной разметке"
assert extended.index.notna().all(), "В расширенной разметке есть пустые ID"

extra_labels = extended.loc[
    extended["weight"].eq(1.0) & ~extended.index.isin(verified.index)
].assign(weight=CHAT_LABEL_WEIGHT)
assert extra_labels.index.isin(X.index).all(), "Для дополнительных меток нет признаков"
assert extra_labels["is_deal"].isin([0, 1]).all(), "Некорректные дополнительные метки"
assert extra_labels.index.intersection(verified.index).empty

print("Дополнительных меток:", len(extra_labels))
display(extra_labels.groupby(["is_deal", "weight"]).size())

Дополнительных меток: 614


is_deal  weight
1        0.1       614
dtype: int64

## 2. Подготовить две модели L2

Для числовых признаков заполняем пропуски медианой и масштабируем значения. Категории заполняем самым частым значением и кодируем отдельными столбцами. Подготовка данных обучается заново на строках каждого фолда.

У модели с дополнительными метками `C=0.3`. У модели на проверенных метках `C=1.0`.

In [17]:
def clean_numbers(frame):
    return frame.astype(float).replace([np.inf, -np.inf], np.nan)


def clean_categories(frame):
    strings = frame.astype("string")
    return strings.astype(object).where(strings.notna(), np.nan)


numeric_pipeline = make_pipeline(
    FunctionTransformer(clean_numbers, feature_names_out="one-to-one"),
    SimpleImputer(strategy="median"),
    StandardScaler(),
)
categorical_pipeline = make_pipeline(
    FunctionTransformer(clean_categories, feature_names_out="one-to-one"),
    SimpleImputer(strategy="most_frequent"),
    OneHotEncoder(handle_unknown="ignore"),
)
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, make_column_selector(dtype_include=["number", "bool"])),
    ("categorical", categorical_pipeline, make_column_selector(dtype_exclude=["number", "bool"])),
])
verified_model = make_pipeline(
    preprocessor,
    LogisticRegression(C=VERIFIED_C, max_iter=2000),
)
chat_model = clone(verified_model).set_params(logisticregression__C=CHAT_C)

## 3. Проверить качество на отложенных объявлениях

В каждом из десяти повторов делим `verified.csv` на пять внешних фолдов. Внутри обучающей части каждого фолда строим отложенные прогнозы на четырёх разбиениях и выбираем порог с максимальной суммой баллов ТЗ. При равенстве выбираем порог ближе к 0,5. После этого проверяем результат на внешнем фолде.

Считаем F1 и CatShareMAE по всем отложенным прогнозам каждого повтора. CatShareMAE — средняя абсолютная ошибка доли сделок по категориям в процентных пунктах. За F1 начисляется до 20 баллов, за CatShareMAE — до 10. Таблица показывает расчёт на проверенной разметке.

In [18]:
blocked_ids = listings.index[listings["close_type"] == "заблокировано"]

In [19]:
def fit_model(model, X, y_train, extra_labels):
    train_labels = pd.concat([
        y_train.to_frame().assign(weight=1.0),
        extra_labels,
    ])
    X_fit = X.loc[train_labels.index]
    return clone(model).fit(
        X_fit,
        train_labels["is_deal"],
        logisticregression__sample_weight=train_labels["weight"],
    )

In [20]:
def fit_ensemble(y_train):
    fitted_chat = fit_model(chat_model, X, y_train, extra_labels)
    fitted_verified = fit_model(verified_model, X, y_train, extra_labels.iloc[:0])
    return fitted_chat, fitted_verified


def deal_probabilities(fitted_models, X_test):
    fitted_chat, fitted_verified = fitted_models
    chat_probabilities = fitted_chat.predict_proba(X_test)[:, 1]
    verified_probabilities = fitted_verified.predict_proba(X_test)[:, 1]
    probabilities = (
        CHAT_MODEL_SHARE * chat_probabilities
        + (1 - CHAT_MODEL_SHARE) * verified_probabilities
    )
    probabilities[X_test.index.isin(blocked_ids)] = 0
    return probabilities

In [21]:
def competition_score(f1, cat_share_mae):
    f1_points = 20 * np.clip((f1 - 0.60) / 0.25, 0, 1)
    share_points = 10 * np.clip((15 - cat_share_mae) / 14, 0, 1)
    return f1_points + share_points


def choose_threshold(y_train):
    inner_cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=INNER_SEED)
    probabilities = np.full(len(y_train), np.nan)

    for train, test in inner_cv.split(y_train, y_train):
        fitted = fit_ensemble(y_train.iloc[train])
        probabilities[test] = deal_probabilities(fitted, X.loc[y_train.iloc[test].index])

    assert np.isfinite(probabilities).all(), "Пропущены внутренние прогнозы"

    # У каждого порога свой столбец прогнозов.
    predicted = probabilities[:, None] > THRESHOLDS
    actual = y_train.to_numpy()[:, None] == 1
    true_positives = (predicted & actual).sum(axis=0)
    f1 = 2 * true_positives / (predicted.sum(axis=0) + actual.sum())

    train_categories = categories.loc[y_train.index]
    category_errors = []
    for category in train_categories.unique():
        in_category = train_categories.eq(category).to_numpy()
        share_error = predicted[in_category].mean(axis=0) - actual[in_category].mean()
        category_errors.append(100 * np.abs(share_error))
    cat_share_mae = np.mean(category_errors, axis=0)

    scores = competition_score(f1, cat_share_mae)
    best = np.flatnonzero(scores == scores.max())
    closest = np.abs(THRESHOLDS[best] - 0.5).argmin()
    return THRESHOLDS[best[closest]]

In [22]:
def evaluate_ensemble(seed):
    outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    fold_predictions = []

    for fold, (train, test) in enumerate(outer_cv.split(y, y), start=1):
        y_train, y_test = y.iloc[train], y.iloc[test]
        threshold = choose_threshold(y_train)
        fitted = fit_ensemble(y_train)
        probabilities = deal_probabilities(fitted, X.loc[y_test.index])

        fold_predictions.append(pd.DataFrame({
            "fold": fold,
            "threshold": threshold,
            "is_deal_true": y_test,
            "is_deal_pred": (probabilities > threshold).astype(int),
        }))

    predictions = pd.concat(fold_predictions).loc[y.index]
    assert predictions.index.equals(y.index), "Нужен один прогноз на каждое проверенное объявление"
    return predictions


def category_deal_shares(predictions):
    return predictions.join(categories).groupby("category")[
        ["is_deal_true", "is_deal_pred"]
    ].mean()

### Метрики по десяти разбиениям

In [23]:
validation_results = []
validation_predictions = {}

for seed in CV_SEEDS:
    checked = evaluate_ensemble(seed)
    validation_predictions[seed] = checked
    shares = category_deal_shares(checked)
    cat_share_mae = 100 * (shares["is_deal_true"] - shares["is_deal_pred"]).abs().mean()
    actual, predicted = checked["is_deal_true"], checked["is_deal_pred"]
    f1 = f1_score(actual, predicted)

    validation_results.append({
        "Разбиение": seed,
        "F1": f1,
        "CatShareMAE, п. п.": cat_share_mae,
        "Precision": precision_score(actual, predicted, zero_division=0),
        "Recall": recall_score(actual, predicted, zero_division=0),
        "Баллы из 30": competition_score(f1, cat_share_mae),
    })

validation_results = pd.DataFrame(validation_results).set_index("Разбиение")
display(validation_results.round(4))
print("Среднее по десяти разбиениям")
display(validation_results.mean().round(4))

,F1,"CatShareMAE, п. п.",Precision,Recall,Баллы из 30
Разбиение,,,,,
11,0.8240,0.8581,0.8205,0.8276,27.9227
23,0.8210,3.5714,0.8319,0.8103,25.8401
42,0.8246,3.6097,0.8393,0.8103,26.1008
71,0.8326,1.1451,0.8291,0.8362,28.5058
101,0.8291,2.0904,0.8220,0.8362,27.5459
13,0.8087,4.1008,0.8158,0.8017,24.4808
29,0.8304,2.8990,0.8611,0.8017,27.0722
59,0.8155,3.3411,0.8120,0.8190,25.5638
83,0.8264,4.0476,0.7937,0.8621,25.9388


Среднее по десяти разбиениям


F1                     0.8266
CatShareMAE, п. п.     2.9428
Precision              0.8306
Recall                 0.8233
Баллы из 30           26.6996
dtype: float64

In [24]:
example_predictions = validation_predictions[42]
print("Разбиение 42. Пороги внешних фолдов")
display(example_predictions.groupby("fold")["threshold"].first())
print("Доли сделок по категориям")
display(category_deal_shares(example_predictions).round(4))

Разбиение 42. Пороги внешних фолдов


fold
1    0.60
2    0.50
3    0.54
4    0.62
5    0.52
Name: threshold, dtype: float64

Доли сделок по категориям


,is_deal_true,is_deal_pred
category,,
Бытовая техника,0.4000,0.4333
Детские товары,0.4889,0.5111
Запчасти и аксессуары,0.3143,0.2857
Мебель и интерьер,0.4571,0.3714
Одежда и обувь,0.2698,0.2540
Спорт и отдых,0.2500,0.2857
Электроника,0.4844,0.4531


## 4. Обучить итоговые модели и получить прогнозы

По отложенным прогнозам на всей проверенной разметке выбираем один порог ансамбля. Обучаем обе модели на доступных им метках и применяем порог к смеси вероятностей для всех объявлений.

In [25]:
threshold = choose_threshold(y)
print("Итоговый порог:", round(threshold, 2))

final_models = fit_ensemble(y)
probabilities = deal_probabilities(final_models, X)
predictions = X.index.to_frame(index=False).assign(
    is_deal=(probabilities > threshold).astype(int),
)

Итоговый порог: 0.42


## 5. Сохранить CSV

Оставляем прогнозы модели для незаблокированных объявлений. Для заблокированных ставим `is_deal = 0`.

In [27]:
submission = predictions.copy()
submission.loc[submission["listing_id"].isin(blocked_ids), "is_deal"] = 0

assert submission.columns.tolist() == ["listing_id", "is_deal"]
assert submission["listing_id"].is_unique
assert set(submission["listing_id"]) == set(listings.index)
assert submission["is_deal"].isin([0, 1]).all()

submission.to_csv(OUTPUT_PATH, index=False)
print(f"Сохранено {len(submission)} объявлений в {OUTPUT_PATH}")
submission.head()

Сохранено 15274 объявлений в submission.csv


,listing_id,is_deal
0,132065,1
1,131440,0
2,117997,1
3,129810,1
4,111780,1
